In [0]:
%sql
CREATE DATABASE SALES_NEW

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-7924277512497684>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', 'CREATE DATABASE SALES_NEW\n')

File /databricks/python/lib/python3.12/site-packages/IPython/core/interactiveshell.py:2541, in InteractiveShell.run_cell_magic(self, magic_name, line, cell)
   2539 with self.builtin_trap:
   2540     args = (magic_arg_s, cell)
-> 2541     result = fn(*args, **kwargs)
   2543 # The code below prevents the output from being displayed
   2544 # when using magics with decorator @output_can_be_silenced
   2545 # when the last Python token in the expression is a ';'.
   2546 if getattr(fn, magic.MAGIC_OUTPUT_CAN_BE_SILENCED, False):

File /databricks/python_shell/lib/dbruntime/sql_magic/sql_magic.py:213, in SqlMagic.sql(self, line, cell)
    206 except BaseException as e:
    207     self.driver_activity_logger.logExecuteCo

In [0]:
%sql
CREATE OR REPLACE TABLE sales_new.Orders (
    OrderID INT,
    OrderDate DATE,
    CustomerID INT,
    CustomerName VARCHAR(100),
    CustomerEmail VARCHAR(100),
    ProductID INT,
    ProductName VARCHAR(100),
    ProductCategory VARCHAR(50),
    RegionID INT,
    RegionName VARCHAR(50),
    Country VARCHAR(50),
    Quantity INT,
    UnitPrice DECIMAL(10,2),
    TotalAmount DECIMAL(10,2)
);

In [0]:
%sql
INSERT INTO sales_new.Orders (OrderID, OrderDate, CustomerID, CustomerName, CustomerEmail, ProductID, ProductName, ProductCategory, RegionID, RegionName, Country, Quantity, UnitPrice, TotalAmount) 
VALUES 
(1, '2024-02-01', 101, 'Alice Johnson', 'alice@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'USA', 2, 800.00, 1600.00),
(2, '2024-02-02', 102, 'Bob Smith', 'bob@example.com', 202, 'Smartphone', 'Electronics', 302, 'Europe', 'Germany', 1, 500.00, 500.00),
(3, '2024-02-03', 103, 'Charlie Brown', 'charlie@example.com', 203, 'Tablet', 'Electronics', 303, 'Asia', 'India', 3, 300.00, 900.00),
(4, '2024-02-04', 101, 'Alice Johnson', 'alice@example.com', 204, 'Headphones', 'Accessories', 301, 'North America', 'USA', 1, 150.00, 150.00),
(5, '2024-02-05', 104, 'David Lee', 'david@example.com', 205, 'Gaming Console', 'Electronics', 302, 'Europe', 'France', 1, 400.00, 400.00),
(6, '2024-02-06', 102, 'Bob Smith', 'bob@example.com', 206, 'Smartwatch', 'Electronics', 303, 'Asia', 'China', 2, 200.00, 400.00),
(7, '2024-02-07', 105, 'Eve Adams', 'eve@example.com', 201, 'Laptop', 'Electronics', 301, 'North America', 'Canada', 1, 800.00, 800.00),
(8, '2024-02-08', 106, 'Frank Miller', 'frank@example.com', 207, 'Monitor', 'Accessories', 302, 'Europe', 'Italy', 2, 250.00, 500.00),
(9, '2024-02-09', 107, 'Grace White', 'grace@example.com', 208, 'Keyboard', 'Accessories', 303, 'Asia', 'Japan', 3, 100.00, 300.00),
(10, '2024-02-10', 104, 'David Lee', 'david@example.com', 209, 'Mouse', 'Accessories', 301, 'North America', 'USA', 1, 50.00, 50.00);

source



In [0]:
%sql
SELECT * FROM sales_new.orders

In [0]:
%sql
CREATE DATABASE orderDWH

Staging_layer


In [0]:
%sql
CREATE OR REPLACE TABLE
orderdwh.stg_sales
AS
select * from sales_new.orders

Transformation_layer

In [0]:
%sql
CREATE VIEW  orderdwh.trans_sales
as
SELECT * FROM orderdwh.stg_sales where Quantity is NOT NULL

In [0]:
%sql
SELECT * FROM orderdwh.trans_sales

OrderID,OrderDate,CustomerID,CustomerName,CustomerEmail,ProductID,ProductName,ProductCategory,RegionID,RegionName,Country,Quantity,UnitPrice,TotalAmount
1,2024-02-01,101,Alice Johnson,alice@example.com,201,Laptop,Electronics,301,North America,USA,2,800.00,1600.00
2,2024-02-02,102,Bob Smith,bob@example.com,202,Smartphone,Electronics,302,Europe,Germany,1,500.00,500.00
3,2024-02-03,103,Charlie Brown,charlie@example.com,203,Tablet,Electronics,303,Asia,India,3,300.00,900.00
4,2024-02-04,101,Alice Johnson,alice@example.com,204,Headphones,Accessories,301,North America,USA,1,150.00,150.00
5,2024-02-05,104,David Lee,david@example.com,205,Gaming Console,Electronics,302,Europe,France,1,400.00,400.00
6,2024-02-06,102,Bob Smith,bob@example.com,206,Smartwatch,Electronics,303,Asia,China,2,200.00,400.00
7,2024-02-07,105,Eve Adams,eve@example.com,201,Laptop,Electronics,301,North America,Canada,1,800.00,800.00
8,2024-02-08,106,Frank Miller,frank@example.com,207,Monitor,Accessories,302,Europe,Italy,2,250.00,500.00
9,2024-02-09,107,Grace White,grace@example.com,208,Keyboard,Accessories,303,Asia,Japan,3,100.00,300.00
10,2024-02-10,104,David Lee,david@example.com,209,Mouse,Accessories,301,North America,USA,1,50.00,50.00


**Customer_Dimensions**

In [0]:
%sql
CREATE or REPLACE TABLE orderdwh.dim_customers (
    CustomerID int,
    CustomerName string,
    CustomerEmail string,
    DimCustomersKey int
)

In [0]:
%sql
CREATE VIEW orderdwh.dim_customers_view
as
SELECT T.*,row_number()over (order by T.CustomerID) as DimCustomersKey From 
(SELECT 
DISTINCT(CustomerID)as CustomerID,
CustomerName,
CustomerEmail 
from orderdwh.trans_sales
)as T


In [0]:
%sql
Select * from orderdwh.dim_customers_view

CustomerID,CustomerName,CustomerEmail,DimCustomersKey
101,Alice Johnson,alice@example.com,1
102,Bob Smith,bob@example.com,2
103,Charlie Brown,charlie@example.com,3
104,David Lee,david@example.com,4
105,Eve Adams,eve@example.com,5
106,Frank Miller,frank@example.com,6
107,Grace White,grace@example.com,7


In [0]:
%sql
INSERT INTO orderdwh.dim_customers
SELECT  * FROM orderdwh.dim_customers_view

num_affected_rows,num_inserted_rows
7,7


In [0]:
%sql
SELECT * from orderdwh.dim_customers


CustomerID,CustomerName,CustomerEmail,DimCustomersKey
101,Alice Johnson,alice@example.com,1
102,Bob Smith,bob@example.com,2
103,Charlie Brown,charlie@example.com,3
104,David Lee,david@example.com,4
105,Eve Adams,eve@example.com,5
106,Frank Miller,frank@example.com,6
107,Grace White,grace@example.com,7


**Product_Dimensions**

In [0]:
%sql
CREATE or REPLACE TABLE orderdwh.dim_products (
    ProductID int,
    ProductName string,
    ProductCategory string,
    DimProductsKey int
)

In [0]:
%sql
CREATE VIEW orderdwh.dim_products_view
as
SELECT T.*,row_number()over (order by T.ProductID) as DimProductsKey From 
(SELECT 
DISTINCT(ProductID)as ProductID,
ProductName,
ProductCategory 
from orderdwh.trans_sales
)as T

In [0]:
%sql
SELECT * from orderdwh.dim_products_view


ProductID,ProductName,ProductCategory,DimProductsKey
201,Laptop,Electronics,1
202,Smartphone,Electronics,2
203,Tablet,Electronics,3
204,Headphones,Accessories,4
205,Gaming Console,Electronics,5
206,Smartwatch,Electronics,6
207,Monitor,Accessories,7
208,Keyboard,Accessories,8
209,Mouse,Accessories,9


In [0]:
%sql
INSERT INTO orderdwh.dim_products
SELECT * FROM orderdwh.dim_products_view;
    
SELECT * from orderdwh.dim_products 

ProductID,ProductName,ProductCategory,DimProductsKey
201,Laptop,Electronics,1
202,Smartphone,Electronics,2
203,Tablet,Electronics,3
204,Headphones,Accessories,4
205,Gaming Console,Electronics,5
206,Smartwatch,Electronics,6
207,Monitor,Accessories,7
208,Keyboard,Accessories,8
209,Mouse,Accessories,9


**Dimesion_Region**

In [0]:
%sql
create table orderdwh.dim_regions(
    RegionID int,
    RegionName string,
    Country string,
    DimRegionsKey int
)

In [0]:
%sql
CREATE VIEW orderdwh.dim_regions_view
as
SELECT T.*,row_number()over (order by T.RegionID) as DimRegionsKey From 
(SELECT 
DISTINCT(RegionID)as RegionID,
RegionName,
Country 
from orderdwh.trans_sales
)as T

In [0]:
%sql
Insert into orderdwh.dim_regions
SELECT * FROM orderdwh.dim_regions_view;

num_affected_rows,num_inserted_rows
8,8


In [0]:
%sql
SELECT  * from orderdwh.dim_regions

RegionID,RegionName,Country,DimRegionsKey
301,North America,Canada,1
301,North America,USA,2
302,Europe,Italy,3
302,Europe,France,4
302,Europe,Germany,5
303,Asia,China,6
303,Asia,India,7
303,Asia,Japan,8


**Date_Dimensions**

In [0]:
%sql
CREATE or REPLACE TABLE orderdwh.dim_time(
    OrderDate date,
    DimTimeKey int
)

In [0]:
%sql
create or replace view orderdwh.dim_time_view
as
SELECT T.*,row_number()over (order by T.OrderDate) as DimTimeKey From 
(SELECT 
DISTINCT(OrderDate)as OrderDate
from orderdwh.trans_sales
)as T

In [0]:
%sql
SELECT * from orderdwh.dim_time_view

OrderDate,DimTimeKey
2024-02-01,1
2024-02-02,2
2024-02-03,3
2024-02-04,4
2024-02-05,5
2024-02-06,6
2024-02-07,7
2024-02-08,8
2024-02-09,9
2024-02-10,10


In [0]:
%sql
delete from orderdwh.dim_time;
Insert Into orderdwh.dim_time
SELECT * FROM orderdwh.dim_time_view;

Select * from orderdwh.dim_time 

OrderDate,DimTimeKey
2024-02-01,1
2024-02-02,2
2024-02-03,3
2024-02-04,4
2024-02-05,5
2024-02-06,6
2024-02-07,7
2024-02-08,8
2024-02-09,9
2024-02-10,10


**Final_dimension_Fact_table**

In [0]:
%sql
Create table orderdwh.FactSales(
OrderID int,
Quantity int,
UnitPrice decimal(10,2),
TotalAmount decimal(10,2),
DimCustomersKey int,
DimProductsKey int,
DimTimeKey int,
DimRegionsKey int
)



**Key_staging**

In [0]:
%sql
with cte_factSales as (SELECT  
F.OrderID,
F.Quantity,
F.UnitPrice,
F.TotalAmount,
C.DimCustomersKey,
P.DimProductsKey,
T.DimTimeKey,
R.DimRegionsKey
FROM orderdwh.trans_sales as F
left JOIN orderdwh.dim_customers as C
ON F.CustomerID=C.CustomerID
left JOIN orderdwh.dim_products as P
ON F.ProductID=P.ProductID
left JOIN orderdwh.dim_time as T
ON F.OrderDate=T.OrderDate
left JOIN orderdwh.dim_regions as R
ON  F.Country=R.Country)



INSERT INTO orderdwh.FactSales
SELECT * FROM cte_factSales;






    

num_affected_rows,num_inserted_rows
10,10


In [0]:
%sql
Select * from orderdwh.FactSales

OrderID,Quantity,UnitPrice,TotalAmount,DimCustomersKey,DimProductsKey,DimTimeKey,DimRegionsKey
1,2,800.00,1600.00,1,1,1,2
2,1,500.00,500.00,2,2,2,5
3,3,300.00,900.00,3,3,3,7
4,1,150.00,150.00,1,4,4,2
5,1,400.00,400.00,4,5,5,4
6,2,200.00,400.00,2,6,6,6
7,1,800.00,800.00,5,1,7,1
8,2,250.00,500.00,6,7,8,3
9,3,100.00,300.00,7,8,9,8
10,1,50.00,50.00,4,9,10,2
